# Lifetime fitting of a LINCam recording with FLIMKit

[FLIMKit](https://github.com/FLIMKit/FLIMKit) opens LINCam `.photons` files
itself, through [photonsfile](https://github.com/alex1075/photonsfile), a
pure-Python reader of the D7 format, and fits them: the decay of all photons
first, then every pixel with the lifetimes found there held.

The recording is the *Convallaria majalis* section of note 2, 284 million
photons. FLIMKit reads a file whole, some 80 bytes a photon, so this one takes
24 GB of memory.

Five steps, all in [`fit.py`](fit.py), which this notebook only calls:

1. read the photons into a (y, x, time) cube;
2. the instrument response, and where the fit starts;
3. fit the decay of all photons, with one to four lifetimes;
4. fit every pixel with the four lifetimes held;
5. the map of the mean lifetime, and the photons over it.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import fit
# FLIMKit sets matplotlib's Agg backend as these load: load them, then undo it.
import flimkit.FLIM.fitters, flimkit.FLIM.irf_tools
%matplotlib inline

FILE = "convallaria.photons"
MIN_PHOTONS = 1000   # per fitted pixel, after 2 x 2 binning

flim, decay = fit.load(FILE, pixels=512, bins=1024)
bin_s = flim.tcspc_res
t = (np.arange(decay.size) + 0.5) * bin_s * 1e9
print(f"{int(decay.sum()):,} photons, {bin_s * 1e12:.1f} ps a bin")

## 1. The photons

FLIMKit bins every photon by its position, 512 × 512 pixels, and by its
arrival after the laser pulse, 1024 bins over the 4096 TAC channels. Images are
drawn the way Photonscore Preview draws them.

In [ ]:
counts = flim.intensity_image().astype(float)
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 5.5),
                                  layout="constrained")
fit.intensity_panel(left, counts)
right.semilogy(t, np.maximum(decay, 0.5))
right.set_xlabel("arrival after the laser pulse, ns")
right.set_ylabel(f"photons per {bin_s * 1e12:.0f} ps")
right.set_title("All photons")

## 2. The instrument response, and where the fit starts

A Gaussian of 80 ps FWHM at the decay's peak, which the fit may shift and
widen. LINCam's response has a core that narrow, and ahead of it a weak
shoulder, a thousandth of the peak and some 1 ns long, which no Gaussian
follows: the fit starts where the decay climbs through half its peak.

In [ ]:
irf = fit.response(decay, bin_s, 0.08)
start = fit.fit_start(decay, bin_s)
print(f"the fit starts at {start:.2f} ns")

fig, ax = plt.subplots(figsize=(10, 4.5), layout="constrained")
ax.semilogy(t, np.maximum(decay, 0.5), ".", ms=3, label="photons")
ax.semilogy(t, irf / irf.max() * decay.max(), label="Gaussian, 80 ps")
ax.axvline(start, color="#CC3333", label="fit starts")
ax.set_xlim(start - 1.5, start + 1.5)
ax.set_ylim(1, decay.max() * 2)
ax.set_xlabel("arrival after the laser pulse, ns")
ax.legend()

## 3. The decay of all photons

`fit_summed` convolves the exponentials with the response, adds a flat
background, and fits them on the Poisson likelihood: differential evolution,
then Levenberg-Marquardt. Each lifetime added cuts the reduced χ² about
tenfold until four, while the mean lifetime hardly moves.

In [ ]:
print(f"{'n':>2} {'lifetimes, ns':<32} {'mean, ns':>9} {'chi2 red.':>10}")
for n in (1, 2, 3):
  _, s = fit.fit_all(decay, bin_s, irf, n, start)
  taus = ", ".join(f"{tau:.2f}" for tau in sorted(s["taus_ns"]))
  print(f"{n:>2} {taus:<32} {s['tau_mean_int_ns']:>9.3f} "
        f"{s['reduced_chi2']:>10.1f}")

NEXP = 4
popt, summary = fit.fit_all(decay, bin_s, irf, NEXP, start)
taus = np.asarray(summary["taus_ns"])
print(f"{NEXP:>2} {', '.join(f'{tau:.2f}' for tau in sorted(taus)):<32} "
      f"{summary['tau_mean_int_ns']:>9.3f} {summary['reduced_chi2']:>10.1f}")

fig, axes = plt.subplot_mosaic([["decay"], ["residual"]],
                               height_ratios=[3, 1], figsize=(10, 7),
                               layout="constrained")
axes["residual"].sharex(axes["decay"])
fit.decay_panel(axes["decay"], axes["residual"], decay, bin_s, summary)

## 4. Every pixel

2 × 2 pixels binned, 256 × 256, and each fitted with the four lifetimes held:
only the amplitudes are free, solved by non-negative least squares over the
same window.

In [ ]:
stack = flim.pixel_stack(binning=2)
maps = fit.fit_pixels(stack, bin_s, irf, popt, summary, NEXP, MIN_PHOTONS)
fitted = np.isfinite(maps["tau_mean_int"])
print(f"{int(fitted.sum()):,} pixels fitted, holding "
      f"{100 * maps['intensity'][fitted].sum() / stack.sum():.0f} % "
      f"of the photons")

## 5. The mean lifetime

The intensity-weighted mean lifetime of every pixel, its brightness the
photons in it, and how the photons spread over it.

In [ ]:
tau = maps["tau_mean_int"]
lo, hi = np.nanpercentile(tau, [1, 99])
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 5.5),
                                  layout="constrained")
fit.map_panel(left, maps["intensity"], tau,
              "Mean lifetime, intensity-weighted", lo, hi, "turbo", "ns")
fit.histogram_panel(right, maps["intensity"], tau, lo, hi, "turbo")

## 6. The maps, saved

One TIFF of 32-bit float planes, their names in its description.

In [ ]:
names = fit.save_maps("convallaria.fit.tif", maps,
                      fit.shares(maps, taus), taus)
print(names)